In [1]:
import pandas as pd

orders = pd.read_csv(r"C:\Users\Faisa\Downloads\Orders_Messy.csv")
customers = pd.read_csv(r"C:\Users\Faisa\Downloads\Customers_Messy.csv")
products = pd.read_csv(r"C:\Users\Faisa\Downloads\Products_Messy.csv")
returns = pd.read_csv(r"C:\Users\Faisa\Downloads\Returns_Messy.csv")
marketing = pd.read_csv(r"C:\Users\Faisa\Downloads\Marketing_Messy.csv")

In [2]:
orders.head()

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status
0,ORD0001,19-04-24,C520,P504,5.0,1500,0%,Debit Card,Cancelled
1,ORD0002,2024-10-08,C461,P503,2.0,1500,5%,Credit card,delivered
2,ORD0003,24/11/2023,C224,P501,10.0,999,0%,UPI,delivered
3,ORD0004,2023-05-30,C152,P501,NaN,Rs1500,0%,COD,Cancelled
4,ORD0005,2025.04.08,C546,P502,2.0,Rs1500,5%,COD,Cancelled


In [3]:
customers.head()

,Customer_ID,Name,City,Signup_Date,Age,Gender
0,C100,User_0,Bangalore,2023.04.29,NaN,M
1,C101,User_1,Bangalore,03-05-25,45.0,Female
2,C102,User_2,Mumbai,2024.12.09,20.0,M
3,C103,User_3,Bangalore,2022.04.29,NaN,F
4,C104,User_4,Delhi,23-08-25,28.0,Male


In [4]:
products.head()

,Product_ID,Category,Cost_Price,Supplier
0,P501,Electronics,1000,ABC Pvt Ltd
1,P502,electronics,1800,XYZ Traders
2,P503,Fashion,600,ABC Pvt Ltd
3,P504,fashion,1200,NaN
4,P505,Home,2500,Global Corp


In [5]:
returns.head()

,Order_ID,Return_Reason
0,ORD5580,Damaged
1,ORD705,Wrong Item
2,ORD4728,Late Delivery
3,ORD4179,Wrong Item
4,ORD2993,Damaged


In [6]:
marketing.head()

,Date,Channel,Spend,Clicks
0,11/06/2022,Facebook,7314,1351
1,09/07/2022,google,31944,716
2,03-04-22,Facebook,20461,1316
3,2025.08.22,facebook,19428,1009
4,2024-05-06,instagram,19769,1370


In [7]:
#  Convert to string + remove extra spaces
orders['Order_Date'] = orders['Order_Date'].astype(str).str.strip()

# Normalize separators (. / → -)
orders['Order_Date'] = (
    orders['Order_Date']
        .str.replace(".", "-", regex=False)
        .str.replace("/", "-", regex=False)
)

#  Convert to datetime (auto detect mixed format)
orders['Order_Date'] = pd.to_datetime(
    orders['Order_Date'],
    errors='coerce',
    format='mixed'
)

In [8]:
#  Convert to string + remove extra spaces
marketing['Date'] = marketing['Date'].astype(str).str.strip()

#  Normalize separators (. / → -)
marketing['Date'] = (
   marketing['Date']
        .str.replace(".", "-", regex=False)
        .str.replace("/", "-", regex=False)
)

#  Convert to datetime (auto detect mixed format)
marketing['_Date'] = pd.to_datetime(
    marketing['Date'],
    errors='coerce',
    format='mixed'
)

In [9]:
#  Convert to string + remove extra spaces
customers['Signup_Date'] = customers['Signup_Date'].astype(str).str.strip()

#Normalize separators (. / → -)
customers['Signup_Date'] = (
    customers['Signup_Date']
        .str.replace(".", "-", regex=False)
        .str.replace("/", "-", regex=False)
)

#  Convert to datetime (auto detect mixed format)
customers['Signup_Date'] = pd.to_datetime(
    customers['Signup_Date'],
    errors='coerce',
    format='mixed'
)

In [10]:
# Clean Unit_Price
orders['Unit_Price'] = (orders['Unit_Price'].astype(str).str.replace("Rs", "", regex=False).str.replace(",", "", regex=False))   # Remove commas.str.strip()                         # Remove extra spaces.astype(float))

In [11]:
# Clean discount
orders['Discount'] = orders['Discount'].str.replace("%","").str.strip()
orders['Discount'] = orders['Discount'].astype(float)/100

In [12]:
# Fix Quantity
orders['Quantity'] = orders['Quantity'].fillna(1)

In [13]:
# Standardize text
orders['Payment_Method'] = orders['Payment_Method'].str.title()
orders['Order_Status'] = orders['Order_Status'].str.title()
products['Category'] = products['Category'].str.title()
customers['City'] = customers['City'].str.title()

In [14]:
# Fill missing Age
customers['Age'] = customers['Age'].fillna(customers['Age'].median())

In [15]:
# Fill missing supplier
products['Supplier'] = products['Supplier'].fillna("Unknown")


In [16]:
# ---------------- MERGE TABLES ----------------

df = orders.merge(customers, on="Customer_ID", how="left") \
           .merge(products, on="Product_ID", how="left")


In [17]:
df

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Name,City,Signup_Date,Age,Gender,Category,Cost_Price,Supplier
0,ORD0001,2024-04-19,C520,P504,5.0,1500,0.00,Debit Card,Cancelled,User_420,Mumbai,2023-02-20,30.0,F,Fashion,1200,Unknown
1,ORD0002,2024-10-08,C461,P503,2.0,1500,0.05,Credit Card,Delivered,User_361,Bangalore,2023-02-08,28.0,F,Fashion,600,ABC Pvt Ltd
2,ORD0003,2023-11-24,C224,P501,10.0,999,0.00,Upi,Delivered,User_124,Delhi,2024-08-20,30.0,Male,Electronics,1000,ABC Pvt Ltd
3,ORD0004,2023-05-30,C152,P501,1.0,1500,0.00,Cod,Cancelled,User_52,Delhi,2024-12-03,28.0,F,Electronics,1000,ABC Pvt Ltd
4,ORD0005,2025-04-08,C546,P502,2.0,1500,0.05,Cod,Cancelled,User_446,Delhi,2024-06-15,45.0,Male,Electronics,1800,XYZ Traders
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,ORD5046,2024-11-09,C371,P502,2.0,1500,0.10,Debit Card,Delivered,User_271,Delhi,2023-01-05,28.0,F,Electronics,1800,XYZ Traders
5046,ORD5047,2024-08-15,C241,P505,1.0,1500,0.10,Debit Card,Cancelled,User_141,Bangalore,2023-06-04,28.0,Male,Home,2500,Global Corp
5047,ORD5048,2023-08-30,C237,P505,10.0,1500,0.00,Debit Card,Delivered,User_137,Mumbai,2025-08-04,45.0,M,Home,2500,Global Corp
5048,ORD5049,2024-08-27,C338,P503,5.0,1500,0.05,Upi,Cancelled,User_238,Mumbai,2025-11-06,28.0,Female,Fashion,600,ABC Pvt Ltd


In [18]:
df['Gender']=df['Gender'].replace("F","Female")
df['Gender']=df['Gender'].replace("M","Male")

In [19]:
df

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Name,City,Signup_Date,Age,Gender,Category,Cost_Price,Supplier
0,ORD0001,2024-04-19,C520,P504,5.0,1500,0.00,Debit Card,Cancelled,User_420,Mumbai,2023-02-20,30.0,Female,Fashion,1200,Unknown
1,ORD0002,2024-10-08,C461,P503,2.0,1500,0.05,Credit Card,Delivered,User_361,Bangalore,2023-02-08,28.0,Female,Fashion,600,ABC Pvt Ltd
2,ORD0003,2023-11-24,C224,P501,10.0,999,0.00,Upi,Delivered,User_124,Delhi,2024-08-20,30.0,Male,Electronics,1000,ABC Pvt Ltd
3,ORD0004,2023-05-30,C152,P501,1.0,1500,0.00,Cod,Cancelled,User_52,Delhi,2024-12-03,28.0,Female,Electronics,1000,ABC Pvt Ltd
4,ORD0005,2025-04-08,C546,P502,2.0,1500,0.05,Cod,Cancelled,User_446,Delhi,2024-06-15,45.0,Male,Electronics,1800,XYZ Traders
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,ORD5046,2024-11-09,C371,P502,2.0,1500,0.10,Debit Card,Delivered,User_271,Delhi,2023-01-05,28.0,Female,Electronics,1800,XYZ Traders
5046,ORD5047,2024-08-15,C241,P505,1.0,1500,0.10,Debit Card,Cancelled,User_141,Bangalore,2023-06-04,28.0,Male,Home,2500,Global Corp
5047,ORD5048,2023-08-30,C237,P505,10.0,1500,0.00,Debit Card,Delivered,User_137,Mumbai,2025-08-04,45.0,Male,Home,2500,Global Corp
5048,ORD5049,2024-08-27,C338,P503,5.0,1500,0.05,Upi,Cancelled,User_238,Mumbai,2025-11-06,28.0,Female,Fashion,600,ABC Pvt Ltd


In [20]:
df['Gender'].isna().sum()

np.int64(17)

In [21]:
df['Gender']=df['Gender'].fillna(df['Gender'].mode()[0])

In [22]:
df['Gender'].isna().sum()

np.int64(0)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order_ID        5050 non-null   object        
 1   Order_Date      5050 non-null   datetime64[ns]
 2   Customer_ID     5040 non-null   object        
 3   Product_ID      5050 non-null   object        
 4   Quantity        5050 non-null   float64       
 5   Unit_Price      5050 non-null   object        
 6   Discount        5050 non-null   float64       
 7   Payment_Method  5050 non-null   object        
 8   Order_Status    5050 non-null   object        
 9   Name            5033 non-null   object        
 10  City            5033 non-null   object        
 11  Signup_Date     5033 non-null   datetime64[ns]
 12  Age             5033 non-null   float64       
 13  Gender          5050 non-null   object        
 14  Category        5050 non-null   object        
 15  Cost

In [24]:
df[df['Name'].isna()]

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Name,City,Signup_Date,Age,Gender,Category,Cost_Price,Supplier
283,ORD0284,2023-05-04,NaN,P501,1.0,1500,0.00,Credit Card,Returned,NaN,NaN,NaT,NaN,Male,Electronics,1000,ABC Pvt Ltd
800,ORD0801,2022-01-04,NaN,P503,10.0,1500,0.10,Debit Card,Returned,NaN,NaN,NaT,NaN,Male,Fashion,600,ABC Pvt Ltd
1048,ORD1049,2022-03-26,NaN,P502,5.0,999,0.05,Credit Card,Cancelled,NaN,NaN,NaT,NaN,Male,Electronics,1800,XYZ Traders
1238,ORD1239,2024-09-20,NaN,P503,10.0,2500,0.15,Upi,Cancelled,NaN,NaN,NaT,NaN,Male,Fashion,600,ABC Pvt Ltd
1245,ORD1246,2025-06-14,NaN,P503,5.0,999,0.15,Upi,Returned,NaN,NaN,NaT,NaN,Male,Fashion,600,ABC Pvt Ltd
1583,ORD1584,2022-11-13,NaN,P503,1.0,1500,0.15,Credit Card,Returned,NaN,NaN,NaT,NaN,Male,Fashion,600,ABC Pvt Ltd
1797,ORD1798,2023-12-05,NaN,P504,10.0,2500,0.10,Credit Card,Cancelled,NaN,NaN,NaT,NaN,Male,Fashion,1200,Unknown
2514,ORD2515,2023-05-14,C999,P505,10.0,1500,0.10,Credit Card,Delivered,NaN,NaN,NaT,NaN,Male,Home,2500,Global Corp
2703,ORD2704,2022-01-16,C999,P501,5.0,999,0.10,Credit Card,Delivered,NaN,NaN,NaT,NaN,Male,Electronics,1000,ABC Pvt Ltd
2894,ORD2895,2023-04-13,NaN,P503,5.0,1500,0.15,Credit Card,Delivered,NaN,NaN,NaT,NaN,Male,Fashion,600,ABC Pvt Ltd


In [25]:
df['Quantity'] = df['Quantity'].astype(int)

In [26]:
df['Total_Amount'] = df['Unit_Price'] * df['Quantity']

In [27]:
df['Unit_Price']=df['Unit_Price'].astype(float)

In [28]:
df['Total_Amount'] = df['Unit_Price'] * df['Quantity']

In [29]:
# Group by whether the Name is Null or not
revenue_check = df.groupby(df['Name'].isnull())['Total_Amount'].sum()

print("Revenue from Matched Customers vs Missing Customers:")
print(revenue_check)

Revenue from Matched Customers vs Missing Customers:
Name
False    31094000.0
True       156985.0
Name: Total_Amount, dtype: float64


In [30]:
# Fill missing Signup_Date with the date of the Order
df['Signup_Date'] = df['Signup_Date'].fillna(df['Order_Date'])
# 1. Fill Name and City so they don't show up as NaN in charts
df['Name'] = df['Name'].fillna('Unknown/Guest')
df['City'] = df['City'].fillna('Unspecified')
df['Age'] = df['Age'].fillna(df['Age'].mean())


In [31]:
df['Total_Revenue'] = df['Total_Amount'] * (1 - df['Discount'])
df['Profit'] = df['Total_Revenue'] - (df['Cost_Price'] * df['Quantity'])

In [32]:
df

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Name,City,Signup_Date,Age,Gender,Category,Cost_Price,Supplier,Total_Amount,Total_Revenue,Profit
0,ORD0001,2024-04-19,C520,P504,5,1500.0,0.00,Debit Card,Cancelled,User_420,Mumbai,2023-02-20,30.0,Female,Fashion,1200,Unknown,7500.0,7500.0,1500.0
1,ORD0002,2024-10-08,C461,P503,2,1500.0,0.05,Credit Card,Delivered,User_361,Bangalore,2023-02-08,28.0,Female,Fashion,600,ABC Pvt Ltd,3000.0,2850.0,1650.0
2,ORD0003,2023-11-24,C224,P501,10,999.0,0.00,Upi,Delivered,User_124,Delhi,2024-08-20,30.0,Male,Electronics,1000,ABC Pvt Ltd,9990.0,9990.0,-10.0
3,ORD0004,2023-05-30,C152,P501,1,1500.0,0.00,Cod,Cancelled,User_52,Delhi,2024-12-03,28.0,Female,Electronics,1000,ABC Pvt Ltd,1500.0,1500.0,500.0
4,ORD0005,2025-04-08,C546,P502,2,1500.0,0.05,Cod,Cancelled,User_446,Delhi,2024-06-15,45.0,Male,Electronics,1800,XYZ Traders,3000.0,2850.0,-750.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,ORD5046,2024-11-09,C371,P502,2,1500.0,0.10,Debit Card,Delivered,User_271,Delhi,2023-01-05,28.0,Female,Electronics,1800,XYZ Traders,3000.0,2700.0,-900.0
5046,ORD5047,2024-08-15,C241,P505,1,1500.0,0.10,Debit Card,Cancelled,User_141,Bangalore,2023-06-04,28.0,Male,Home,2500,Global Corp,1500.0,1350.0,-1150.0
5047,ORD5048,2023-08-30,C237,P505,10,1500.0,0.00,Debit Card,Delivered,User_137,Mumbai,2025-08-04,45.0,Male,Home,2500,Global Corp,15000.0,15000.0,-10000.0
5048,ORD5049,2024-08-27,C338,P503,5,1500.0,0.05,Upi,Cancelled,User_238,Mumbai,2025-11-06,28.0,Female,Fashion,600,ABC Pvt Ltd,7500.0,7125.0,4125.0


In [33]:
df['Profit'].max()

19000.0

In [34]:
df.drop('Total_Amount',axis=1,inplace=True)

In [35]:
df

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Name,City,Signup_Date,Age,Gender,Category,Cost_Price,Supplier,Total_Revenue,Profit
0,ORD0001,2024-04-19,C520,P504,5,1500.0,0.00,Debit Card,Cancelled,User_420,Mumbai,2023-02-20,30.0,Female,Fashion,1200,Unknown,7500.0,1500.0
1,ORD0002,2024-10-08,C461,P503,2,1500.0,0.05,Credit Card,Delivered,User_361,Bangalore,2023-02-08,28.0,Female,Fashion,600,ABC Pvt Ltd,2850.0,1650.0
2,ORD0003,2023-11-24,C224,P501,10,999.0,0.00,Upi,Delivered,User_124,Delhi,2024-08-20,30.0,Male,Electronics,1000,ABC Pvt Ltd,9990.0,-10.0
3,ORD0004,2023-05-30,C152,P501,1,1500.0,0.00,Cod,Cancelled,User_52,Delhi,2024-12-03,28.0,Female,Electronics,1000,ABC Pvt Ltd,1500.0,500.0
4,ORD0005,2025-04-08,C546,P502,2,1500.0,0.05,Cod,Cancelled,User_446,Delhi,2024-06-15,45.0,Male,Electronics,1800,XYZ Traders,2850.0,-750.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,ORD5046,2024-11-09,C371,P502,2,1500.0,0.10,Debit Card,Delivered,User_271,Delhi,2023-01-05,28.0,Female,Electronics,1800,XYZ Traders,2700.0,-900.0
5046,ORD5047,2024-08-15,C241,P505,1,1500.0,0.10,Debit Card,Cancelled,User_141,Bangalore,2023-06-04,28.0,Male,Home,2500,Global Corp,1350.0,-1150.0
5047,ORD5048,2023-08-30,C237,P505,10,1500.0,0.00,Debit Card,Delivered,User_137,Mumbai,2025-08-04,45.0,Male,Home,2500,Global Corp,15000.0,-10000.0
5048,ORD5049,2024-08-27,C338,P503,5,1500.0,0.05,Upi,Cancelled,User_238,Mumbai,2025-11-06,28.0,Female,Fashion,600,ABC Pvt Ltd,7125.0,4125.0


In [36]:
df.to_csv(r"C:\Users\Faisa\Downloads\Clean_Ecommerce_Data.csv", index=False)

In [37]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Faisa\Downloads\Clean_Ecommerce_Data.csv")

In [38]:
df

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Name,City,Signup_Date,Age,Gender,Category,Cost_Price,Supplier,Total_Revenue,Profit
0,ORD0001,2024-04-19,C520,P504,5,1500.0,0.00,Debit Card,Cancelled,User_420,Mumbai,2023-02-20,30.0,Female,Fashion,1200,Unknown,7500.0,1500.0
1,ORD0002,2024-10-08,C461,P503,2,1500.0,0.05,Credit Card,Delivered,User_361,Bangalore,2023-02-08,28.0,Female,Fashion,600,ABC Pvt Ltd,2850.0,1650.0
2,ORD0003,2023-11-24,C224,P501,10,999.0,0.00,Upi,Delivered,User_124,Delhi,2024-08-20,30.0,Male,Electronics,1000,ABC Pvt Ltd,9990.0,-10.0
3,ORD0004,2023-05-30,C152,P501,1,1500.0,0.00,Cod,Cancelled,User_52,Delhi,2024-12-03,28.0,Female,Electronics,1000,ABC Pvt Ltd,1500.0,500.0
4,ORD0005,2025-04-08,C546,P502,2,1500.0,0.05,Cod,Cancelled,User_446,Delhi,2024-06-15,45.0,Male,Electronics,1800,XYZ Traders,2850.0,-750.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5045,ORD5046,2024-11-09,C371,P502,2,1500.0,0.10,Debit Card,Delivered,User_271,Delhi,2023-01-05,28.0,Female,Electronics,1800,XYZ Traders,2700.0,-900.0
5046,ORD5047,2024-08-15,C241,P505,1,1500.0,0.10,Debit Card,Cancelled,User_141,Bangalore,2023-06-04,28.0,Male,Home,2500,Global Corp,1350.0,-1150.0
5047,ORD5048,2023-08-30,C237,P505,10,1500.0,0.00,Debit Card,Delivered,User_137,Mumbai,2025-08-04,45.0,Male,Home,2500,Global Corp,15000.0,-10000.0
5048,ORD5049,2024-08-27,C338,P503,5,1500.0,0.05,Upi,Cancelled,User_238,Mumbai,2025-11-06,28.0,Female,Fashion,600,ABC Pvt Ltd,7125.0,4125.0


In [39]:
df["Order_ID"] = df["Order_ID"].str.upper()

In [40]:
df['Order_ID'].tail()

5045    ORD5046
5046    ORD5047
5047    ORD5048
5048    ORD5049
5049    ORD5050
Name: Order_ID, dtype: object

In [41]:
df['Profit'].head(10)

0    1500.00
1    1650.00
2     -10.00
3     500.00
4    -750.00
5   -6125.00
6    -500.00
7   -1754.25
8     750.00
9     325.00
Name: Profit, dtype: float64

In [42]:
df.to_csv(r"C:\Users\Faisa\Downloads\Clean_Ecommerce_Data.csv", index=False)

In [43]:
df.columns

Index(['Order_ID', 'Order_Date', 'Customer_ID', 'Product_ID', 'Quantity',
       'Unit_Price', 'Discount', 'Payment_Method', 'Order_Status', 'Name',
       'City', 'Signup_Date', 'Age', 'Gender', 'Category', 'Cost_Price',
       'Supplier', 'Total_Revenue', 'Profit'],
      dtype='object')

In [44]:
df['Customer_ID'].value_counts()<1

Customer_ID
C130    False
C474    False
C424    False
C174    False
C229    False
        ...  
C577    False
C448    False
C493    False
C464    False
C347    False
Name: count, Length: 501, dtype: bool

In [45]:
df['Supplier'].isna().sum()

np.int64(0)